# Evaluating Numerical Expressions

`2 * (3 + 4) ** 2` is trivial for a person to read and surprisingly awkward for a machine. The difficulty is that infix notation, where operators sit between their operands, encodes precedence and grouping positionally. A processor reading left to right cannot know whether to apply the `*` until it has seen everything after it, and parentheses can nest arbitrarily deep.

The standard solution is to stop reading left to right. Postfix notation (`3 4 + 2 ** 2 *`) places each operator after its operands, which removes the need for precedence rules and parentheses entirely: the expression can be evaluated in a single forward pass. Converting from one form to the other is Dijkstra's shunting-yard algorithm, and both halves, the conversion and the evaluation, run on a stack.

This project builds that in three stages:

1. A `Stack` class, on top of the linked list from earlier in the course
2. `evaluate_postfix()`, evaluating an expression already in postfix form
3. `infix_to_postfix()`, the shunting-yard conversion, and `evaluate()` joining the two

The end result is a function that takes a string and returns a number, doing the work Python's own parser does before any arithmetic happens.

In [1]:
from linked_list import LinkedList

class Stack(LinkedList):

    def push(self, data):
        self.append(data)

    def peek(self):
        return self.tail.data

    def pop(self):
        ret = self.tail.data
        if self.length == 1:
            self.tail = self.head = None
        else:
            self.tail = self.tail.prev
            self.tail.next = None
        self.length -= 1
        return ret

## Tokenizing

Postfix notation places the operator after its operands, so `1 + 2` becomes `1 2 +`. That ordering means an expression can be evaluated in a single left to right pass over a stack: push numbers as they appear, and when an operator turns up, pop the top two values, apply it, and push the result back. After the last token, one value remains on the stack and that is the answer. No precedence rules and no parentheses are needed, because the order is already encoded in the sequence.

Before any of that, the expression string has to become a list of tokens. Assuming every element is space separated, `str.split()` is the whole job.

That assumption is doing real work. A production tokenizer would scan character by character to handle `1+2` and `(1 + 2)*3`, which is a meaningfully harder problem. Requiring spaces trades that away to keep the focus on the stack algorithm, at the cost of rejecting most expressions a user would actually type.

In [2]:
def tokenize(expression):
    return expression.split()

print(tokenize("12 2 4 + / 21 *"))

['12', '2', '4', '+', '/', '21', '*']


## Processing Operators

Each operator gets its own function, and all five follow the same three steps: pop twice, apply the operator, push the result.

The order of those two pops is the thing to get right. A stack returns the most recently pushed element first, so the first `pop()` gives the operand that appeared *last* in the expression, and the second gives the one before it:

```python
result = second_to_top - top   # correct
result = top - second_to_top   # wrong, operands reversed
```

For `4 6 -` the stack holds `[4, 6]`, so `top` is 6 and `second_to_top` is 4. The expression means 4 minus 6, which is `second_to_top - top`.

This only matters for the non-commutative operators. Getting it wrong in `process_plus` or `process_times` produces the right answer anyway, which is precisely why it is worth being deliberate about: the bug would sit undetected in three of the five functions and only surface on subtraction, division and exponentiation.

In [3]:
def process_minus(stack):
    top = stack.pop()
    second_to_top = stack.pop()
    result = second_to_top - top
    stack.push(result)

def process_plus(stack):
    top = stack.pop()
    second_to_top = stack.pop()
    result = second_to_top + top
    stack.push(result)

def process_times(stack):
    top = stack.pop()
    second_to_top = stack.pop()
    result = second_to_top * top
    stack.push(result)

def process_divide(stack):
    top = stack.pop()
    second_to_top = stack.pop()
    result = second_to_top / top
    stack.push(result)

def process_pow(stack):
    top = stack.pop()
    second_to_top = stack.pop()
    result = second_to_top ** top
    stack.push(result)

## Evaluating Postfix Expressions

`evaluate_postfix()` assembles the pieces. It tokenizes the expression, then walks the tokens left to right, dispatching each one:

1. Initialize an empty stack
2. Tokenize the expression
3. For each token: if it is an operator, call the matching `process_` function; otherwise convert it to `float` and push it
4. Return the single value left on the stack

The `else` branch is worth noticing. Anything that is not one of the five operators is assumed to be a number, so a typo becomes a `ValueError` from `float()` rather than a clear message about an unrecognised token. Fine for controlled input, but it means the function trusts its caller.

Two other assumptions are unchecked. A malformed expression can leave more than one value on the stack, and the function returns the top of it without noticing, so `1 2 3 +` quietly returns 5 and discards the 1. An expression with too few operands fails inside `pop()` instead. Both are correct behaviour for well-formed input and silent or confusing for anything else.

The whole thing is a single pass: $O(n)$ in the number of tokens, with no backtracking and no precedence logic, because postfix has already encoded the order of operations in the sequence itself.

In [4]:
def evaluate_postfix(expression):
    tokens = tokenize(expression)
    stack = Stack()
    for token in tokens:
        if token == "+":
            process_plus(stack)
        elif token == "-":
            process_minus(stack)
        elif token == "*":
            process_times(stack)
        elif token == "/":
            process_divide(stack)
        elif token == "**":
            process_pow(stack)
        else:
            # The token is not an operator so it must be a number
            stack.push(float(token))
    return stack.pop()

In [5]:
# Testing the Implementation.

# Expected Results:
# -2.0
# 8.0
# 0.0
# 2.0
# 11.25
# 45.0
# 42.0
# 4.0
# 2.0

expressions = [
    "4 6 -",
    "4 1 2 9 3 / * + 5 - *",
    "1 2 + 3 -",
    "1 2 - 3 +",
    "10 3 5 * 16 4 - / +",
    "5 3 4 2 - ** *",
    "12 2 4 + / 21 *",
    "1 1 + 2 **",
    "1 1 2 ** +"
]

for expression in expressions:
    print(evaluate_postfix(expression))

-2.0
8.0
0.0
2.0
11.25
45.0
42.0
4.0
2.0


## Precedence

Converting infix to postfix needs a way to compare two operators, so precedence is stored as a dictionary mapping each operator to an integer. Higher binds tighter:

| Operator | Precedence |
|---|---|
| `+` `-` | 1 |
| `*` `/` | 2 |
| `**` | 3 |

The values themselves carry no meaning; only their order does. Any increasing sequence would work identically, since every use is a comparison rather than arithmetic.

The dictionary also doubles as the set of recognised operators. `token in precedence` is how `infix_to_postfix()` decides whether a token is an operator at all, which means adding a new operator is a single entry rather than a change in two places.

One thing it does not capture is associativity. Precedence says `**` binds tighter than `*`, but not that `2 ** 3 ** 2` should group right to left as `2 ** (3 ** 2)` rather than left to right. Python evaluates it the first way, giving 512; this implementation gives 64. A complete converter would track associativity alongside precedence and use a strict `>` rather than `>=` when comparing equal-precedence right-associative operators.

In [6]:
precedence = {
    "+": 1,
    "-": 1,
    "*": 2,
    "/": 2,
    "**": 3
}

print(precedence["/"] < precedence["-"])
print(precedence["+"] < precedence["*"])
print(precedence["+"] < precedence["-"])
print(precedence["/"] < precedence["**"])

False
True
False
True


## Processing Tokens in Infix to Postfix Conversion

Four token types, four handlers. Together they are the whole algorithm.

**Opening parenthesis.** Pushed onto the stack as a token in its own right. It carries no arithmetic meaning; it is a marker that a closing parenthesis will later unwind to.

**Closing parenthesis.** Pops operators into the output until it reaches that marker, then discards the marker itself with a final `pop()`. This is what forces everything inside the brackets out before anything outside them, and it is why neither parenthesis ever appears in the postfix result.

**Operator.** The only handler with real logic. Before pushing, it drains any operator already on the stack whose precedence is greater than or equal to the incoming one, since those bind at least as tightly and must be applied first. Two guards sit in front of that comparison:

- `len(stack) > 0` because `peek()` reads `self.tail.data` and raises `AttributeError` on an empty stack rather than returning anything useful - `stack.peek() in precedence` because the top may be an opening parenthesis, which has no precedence entry and would raise `KeyError`

Python evaluates `and` left to right and stops at the first false condition, so the ordering is not stylistic. Both guards must precede the lookup they protect.

**Number.** Appended straight to the output. Numbers never wait, which is why operands keep their relative order between the two notations while operators move.

Note that `process_number` takes only the output list, while the others take the stack. The asymmetry is the point: numbers pass through, and only operators and parentheses need somewhere to be held.

In [7]:
def process_opening_parenthesis(stack):
    stack.push("(")

def process_closing_parenthesis(stack, postfix):
    while stack.peek() != "(":
        postfix.append(stack.pop())
    stack.pop()

def process_operator(stack, postfix, operator):
    while len(stack) > 0 and stack.peek() in precedence and precedence[stack.peek()] >= precedence[operator]:
        postfix.append(stack.pop())
    stack.push(operator)

def process_number(postfix, number):
    postfix.append(number)

# The Shunting-yard Algorithm

1. We start by splitting the expression into tokens using the `tokenize()` function.
2. We initialize an empty stack.
3. We initialize and empty postfix token list.
4. Iterate over all tokens and for each of them:
    - If the token is `"("` we call the `process_opening_parenthesis()` function.
    - If the token is `")"` we call the `process_closing_parenthesis()` function.
    - If the token is an operator we call the `process_operator()` function.
    - Otherwise, the token is a number and we call the `process_number()` function.
5. After processing all tokens, we use a while loop to pop the remaining stack element into the postfix token list.
6. Use the `str.join()` method to convert the postfix token list into a string.

In [8]:
def infix_to_postfix(expression):
    tokens = tokenize(expression)
    stack = Stack()
    postfix = []
    for token in tokens:
        if token == "(":
            process_opening_parenthesis(stack)
        elif token == ")":
            process_closing_parenthesis(stack, postfix)
        elif token in precedence:
            process_operator(stack, postfix, token)
        else:
            process_number(postfix, token)
    while len(stack) > 0:
        postfix.append(stack.pop())
    return " ".join(postfix)

# Evaluating Infix Expressions

In [9]:
def evaluate(expression):
    postfix_expression = infix_to_postfix(expression)
    return evaluate_postfix(postfix_expression)

In [10]:
# Testing the Implementation.

# Expected Results:
# 2.0
# 0.0
# 8.0
# 11.25
# 256.0
# 65536.0
# 0.5
# 9.0
# 1.0

expressions = [
    "1 + 1",
    "1 * ( 2 - ( 1 + 1 ) )",
    "4 * ( 1 + 2 * ( 9 / 3 ) - 5 )",
    "10 + 3 * 5 / ( 16 - 4 * 1 )",
    "2 * 2 * 2 * 2 * 2 * 2 * 2 * 2",
    "2 ** 2 ** 2 ** 2 ** 2",
    "( 1 - 2 ) / ( 3 - 5 )",
    "9 / 8 * 8",
    "64 / ( 8 * 8 )",
]

for expression in expressions:
    print(evaluate(expression))

2.0
0.0
8.0
11.25
256.0
65536.0
0.5
9.0
1.0
